In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('fundamentals_groupby_join') \
    .config("spark.jars", "/home/kantundpeterpan/projects/zoomcamp/zcde_space/week5/gcs-connector-hadoop3-latest.jar") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .config("spark.driver.memory", "6g") \
    .config("spark.memory.offHeap.enabled", True) \
    .config("spark.memory.offHeap.size","16g") \
    .getOrCreate()

25/03/06 18:13:08 WARN Utils: Your hostname, mystuff resolves to a loopback address: 127.0.1.1; using 193.168.147.155 instead (on interface eth0)
25/03/06 18:13:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/03/06 18:13:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
# Configure GCS authentication
spark.conf.set("google.cloud.auth.service.account.enable", "true")
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", 
                                     "/home/kantundpeterpan/projects/zoomcamp/zcde_space/week1/3_intro_terraform/workspaceaddon-436615-4bcf737409b7.json")
spark._jsc.hadoopConfiguration().set('fs.gs.impl', 'com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem')

In [4]:
bucket_url = 'gs://workspaceaddon-436615'

In [26]:
df_green = spark.read.parquet(bucket_url + '/green/*/*') \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")

In [9]:
df_green.registerTempTable("green")

/home/kantundpeterpan/projects/zoomcamp/zcde_space/week5/.venv/lib/python3.11/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [18]:
df_green_revenue = spark.sql("""
SELECT 
    -- Reveneue grouping 
    date_trunc('hour', pickup_datetime) AS hour,
    PULocationID AS revenue_zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
  pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
ORDER BY
    1, 2
    
""")

In [19]:
df_green_revenue.show()

+-------------------+------------+------------------+--------------+
|               hour|revenue_zone|            amount|number_records|
+-------------------+------------+------------------+--------------+
|2020-01-01 00:00:00|           7| 769.7299999999998|            45|
|2020-01-01 00:00:00|          17|195.03000000000003|             9|
|2020-01-01 00:00:00|          18|               7.8|             1|
|2020-01-01 00:00:00|          22|              15.8|             1|
|2020-01-01 00:00:00|          24|              87.6|             3|
|2020-01-01 00:00:00|          25|             531.0|            26|
|2020-01-01 00:00:00|          29|              61.3|             1|
|2020-01-01 00:00:00|          32| 68.94999999999999|             2|
|2020-01-01 00:00:00|          33|317.27000000000004|            11|
|2020-01-01 00:00:00|          35|            129.96|             5|
|2020-01-01 00:00:00|          36|295.34000000000003|            11|
|2020-01-01 00:00:00|          37|

In [ ]:
df_green_revenue.write.parquet(
    bucket_url + '/data/report/revenue/green'
)

25/03/06 16:21:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [11]:
df_yellow = spark.read.parquet(bucket_url + '/yellow/*/*') \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")

In [12]:
df_yellow.registerTempTable("yellow")

/home/kantundpeterpan/projects/zoomcamp/zcde_space/week5/.venv/lib/python3.11/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [13]:
df_yellow_revenue = spark.sql("""
SELECT 
    -- Reveneue grouping 
    date_trunc('hour', pickup_datetime) AS hour,
    PULocationID AS revenue_zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
  pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [14]:
df_yellow_revenue.write.parquet(
    bucket_url + '/data/report/revenue/yellow',
    mode = 'overwrite'
)

In [ ]:
spark.stop()

# JOIN operations


In [2]:
# from pyspark.pandas import SparkSession

In [10]:
df_yellow_revenue = spark.read.parquet(bucket_url + '/data/report/revenue/yellow')
df_green_revenue = spark.read.parquet(bucket_url + '/data/report/revenue/green')

In [11]:
df_join = df_green_revenue \
    .withColumnRenamed("amount", "green_amount") \
    .withColumnRenamed("number_records", "green_number_records") \
    .join(
    df_yellow_revenue,
    # lsuffix = "_green",
    on = ['hour', 'revenue_zone'],
    how = 'outer'
)

In [12]:
df_join.show()

+-------------------+------------+------------------+--------------------+------------------+--------------+
|               hour|revenue_zone|      green_amount|green_number_records|            amount|number_records|
+-------------------+------------+------------------+--------------------+------------------+--------------+
|2020-01-01 00:00:00|          22|              15.8|                   1|              NULL|          NULL|
|2020-01-01 00:00:00|          25|             531.0|                  26|            324.35|            16|
|2020-01-01 00:00:00|          55|            129.29|                   4|              NULL|          NULL|
|2020-01-01 00:00:00|          56|             99.69|                   3|              18.1|             2|
|2020-01-01 00:00:00|          60|            160.04|                   6|57.620000000000005|             2|
|2020-01-01 00:00:00|          61|            526.71|                  17|            146.64|             3|
|2020-01-01 00:00:0

In [5]:
import pandas as pd

https://docs.databricks.com/gcp/en/pandas/pyspark-pandas-conversion

In [17]:
df_zones = spark.createDataFrame( 
    pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
)

In [19]:
df_zones.write.parquet(bucket_url + "/data/zones", mode = 'overwrite')

In [20]:
df_results = df_join.join(
    df_zones,
    df_join.revenue_zone == df_zones.LocationID
)

In [25]:
df_results.drop("LocationID", 'zone').write.parquet(bucket_url + '/tmp/revenue_zones')